https://www.kaggle.com/competitions/essay-gap-aicc-round-2

In [ ]:
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from torch import nn

In [ ]:
train = pd.read_csv("/kaggle/input/competitions/essay-gap-aicc-round-2/essay-gap/train.csv")
test = pd.read_csv("/kaggle/input/competitions/essay-gap-aicc-round-2/essay-gap/test.csv")

In [ ]:
train

In [ ]:
import transformers as T
from transformers import AutoProcessor, AutoModelForSequenceClassification

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available else 'cpu'
model_name = "bert-base-uncased"
tokenizer = AutoProcessor.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels = 1).to(device)

In [ ]:
example = train.iloc[1,1]
inputs = tokenizer(example, return_tensors = 'pt', truncation = True, padding = 'max_length', max_length = 128)

In [ ]:
print(inputs.input_ids.shape)
print(inputs.attention_mask.shape)

In [ ]:
inputs = inputs.to(device)
with torch.no_grad():
    outputs = model(input_ids = inputs.input_ids, attention_mask = inputs.attention_mask)
outputs

In [ ]:
outputs.logits

In [ ]:
model

In [ ]:
class essaydataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data
        self.before = data['before'].tolist()
        self.after =  data['after'].tolist()
        self.tokenizer = tokenizer
        self.columns = data.columns
        self.opt = np.vstack((np.array(data['opt_0'].tolist()), np.array(data['opt_1'].tolist()), np.array(data['opt_2'].tolist()), np.array(data['opt_3'].tolist())))
    def __len__(self):
        return len(self.before)
    def __getitem__(self, x):
        input_ids = []
        attention_mask = []
        for i in range(4):
            text = self.before[x] + " [SEP] " + self.opt[i, x] + " [SEP] " + self.after[x]
            inputs = self.tokenizer(text, padding = 'max_length', truncation = True, max_length = 128)
            input_ids.append(inputs.input_ids)
            attention_mask.append(inputs.attention_mask)
        if 'label' in self.columns:
            return {
                'input_ids':torch.tensor(input_ids),
                'attention_mask': torch.tensor(attention_mask),
                'label': torch.tensor(self.data.iloc[x, 7])
            }
        else:
            return {
                'input_ids': torch.tensor(input_ids),
                'attention_mask': torch.tensor(attention_mask)
            }
            
        

In [ ]:
dataset = essaydataset(train, tokenizer)

In [ ]:
first = dataset[2]

In [ ]:
print(first['input_ids'].shape, first['label'])

In [ ]:
loader = DataLoader(dataset, batch_size = 8)

In [ ]:
lr = 2e-5
epochs = 5
from tqdm import tqdm

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr = lr)
for x in range(epochs):
    model.train()
    pbar = tqdm(loader)
    losses = []
    for data in pbar:
        optimizer.zero_grad()
        input_ids = data['input_ids']
        attention_mask = data['attention_mask']
        label = data['label']
        input_ids = input_ids.view(8*4, -1)
        attention_mask = attention_mask.view(8*4, -1)
        outputs = model(input_ids = input_ids.to(device), attention_mask = attention_mask.to(device))
        outputs = outputs.logits.view(8, 4).cpu()
        loss = criterion(outputs, label)
        loss.backward()
        losses.append(loss.item())
        optimizer.step()
    print(sum(losses)/len(losses))

In [ ]:
test_dataset = essaydataset(test, tokenizer)
test_loader = DataLoader(test_dataset, batch_size = 8)
model.eval()
answers = np.array([])
with torch.no_grad():
    pbar = tqdm(test_loader)
    for x in pbar:
        attention_mask = x['attention_mask']
        input_ids = x['input_ids']
        input_ids = input_ids.view(4*8, -1)
        attention_mask = attention_mask.view(4*8, -1)
        outputs = model(input_ids = input_ids.to(device), attention_mask = attention_mask.to(device))
        outputs = outputs.logits.view(8, 4).cpu()
        answers = np.hstack((answers, np.argmax(outputs, axis = 1)))
        

In [ ]:
submission = pd.DataFrame({
    'sampleID': test['sampleID'],
    'answer': answers
})

In [ ]:
submission.to_csv('submission.csv', index = False)

In [ ]:
submission